## Q1:Connecting to new SQL Database

Jupytext Installation Reason: <br>
Installing and creating jupytext is necessary as we link this .py file with py:percent so that we can link this to a .ipynb file which will be a clone to this file. Any modification made there in notebook or in python script will result in update on both the files. This way we can do markdown and piece wise coding in ipynb and we will have the same ready with the python file in parallel. This we can acheive best of both world. 

In [1]:
import sqlite3
import pandas as pd
import numpy as np
import os 
import warnings
warnings.filterwarnings("ignore")

In [2]:
sql_connection=sqlite3.connect('nyflight.db')

## Q2: Exporting CSV to Database

In [3]:
airlines=pd.read_csv('nycflights13_airlines.csv.gz',compression='gzip', comment='#')
airports=pd.read_csv('nycflights13_airports.csv.gz',compression='gzip', comment='#')
flights=pd.read_csv('nycflights13_flights.csv.gz',compression='gzip', comment='#')
planes=pd.read_csv('nycflights13_planes.csv.gz',compression='gzip', comment='#')
weather=pd.read_csv('nycflights13_weather.csv.gz',compression='gzip', comment='#')

Using comment ="#" in the read_csv, igonres the irrelevant lines in the gz file takes only the relevant lines in the dataset. 

Writing the dataframes into the sql database as tables. 

In [4]:
airlines.to_sql('Airlines', sql_connection, if_exists='replace', index=False)
airports.to_sql('Airports', sql_connection, if_exists='replace', index=False)
flights.to_sql('Flights', sql_connection, if_exists='replace', index=False)
planes.to_sql('Planes', sql_connection, if_exists='replace', index=False)
weather.to_sql('Weather', sql_connection, if_exists='replace', index=False)

26130

In [5]:
sql_connection.close() #Closing the connection 

## Q3: SQL Explanation

In [7]:
sql_connection=sqlite3.connect('nyflight.db') #connecting back to perform tasks

### Subtask-1

In [8]:
task1_sql =pd.read_sql_query('''  SELECT DISTINCT engine FROM planes  ''', sql_connection)
task1_sql

,engine
0,Turbo-fan
1,Turbo-jet
2,Reciprocating
3,4 Cycle
4,Turbo-shaft
5,Turbo-prop


Using the pd.read_sql_query function to execute an SQL query that retrieves distinct engine types from the planes table in a connected database. <br> 
>The query employs the SELECT DISTINCT statement to ensure that only unique engine values are included in the result. <br>
This query's output is stored in a Pandas DataFrame named task1_sql, which can be further analyzed or processed in Python. <br> 
By leveraging SQL and Pandas together, this approach enables efficient extraction of unique values directly from a database.

In [ ]:
task1_pd=(planes[['engine']].drop_duplicates()
          .reset_index(drop=True))
task1_pd

,engine
0,Turbo-fan
1,Turbo-jet
2,Reciprocating
3,4 Cycle
4,Turbo-shaft
5,Turbo-prop


Selecting the engine column from the planes DataFrame and uses the drop_duplicates() method to remove duplicate entries, ensuring only unique engine types are retained. <br>
The reset_index(drop=True) method resets the index of the resulting DataFrame, dropping the old index to produce a clean, sequential index. The resulting DataFrame, named task1_pd, contains a list of unique engine values from the planes DataFrame. <br>
This approach efficiently extracts distinct values using Pandas without the need for SQL.

In [10]:
print(pd.testing.assert_frame_equal(task1_sql,task1_pd))

None


### Subtask-2

In [11]:
task2_sql =pd.read_sql_query('''  SELECT DISTINCT type, engine FROM Planes  ''', sql_connection)
task2_sql

,type,engine
0,Fixed wing multi engine,Turbo-fan
1,Fixed wing multi engine,Turbo-jet
2,Fixed wing single engine,Reciprocating
3,Fixed wing multi engine,Reciprocating
4,Fixed wing single engine,4 Cycle
5,Rotorcraft,Turbo-shaft
6,Fixed wing multi engine,Turbo-prop


Using the pd.read_sql_query function to execute an SQL query that retrieves distinct pairs of type and engine values from the Planes table. 
>The query uses the SELECT DISTINCT statement to ensure that only unique combinations of the type and engine columns are included in the result.<br>
The query's output is stored in a Pandas DataFrame named task2_sql, allowing further analysis or manipulation in Python. 

In [12]:
task2_pd=(planes[['type','engine']].drop_duplicates()
          .reset_index(drop=True))
task2_pd

,type,engine
0,Fixed wing multi engine,Turbo-fan
1,Fixed wing multi engine,Turbo-jet
2,Fixed wing single engine,Reciprocating
3,Fixed wing multi engine,Reciprocating
4,Fixed wing single engine,4 Cycle
5,Rotorcraft,Turbo-shaft
6,Fixed wing multi engine,Turbo-prop


The code extracts unique combinations of the type and engine columns from the planes DataFrame and stores them in a new DataFrame called task2_pd. <br>
>First, it selects only the type and engine columns using planes[['type', 'engine']]. Then, it removes any duplicate rows in these columns using .drop_duplicates(), ensuring each combination appears only once. <br>
To clean up the indexing after dropping duplicates, .reset_index(drop=True) is used, which resets the index to a sequential order starting from 0, while drop=True prevents the old index from being added as a separate column. Finally, the resulting DataFrame, task2_pd, contains distinct type and engine pairs with clean indexing, making it ready for further use or analysis.

In [13]:
print(pd.testing.assert_frame_equal(task2_sql,task2_pd))

None


### Subtask-3

In [14]:
task3_sql =pd.read_sql_query('''  SELECT COUNT(*), engine FROM Planes GROUP BY engine  ''', sql_connection)
task3_sql

,COUNT(*),engine
0,2,4 Cycle
1,28,Reciprocating
2,2750,Turbo-fan
3,535,Turbo-jet
4,2,Turbo-prop
5,5,Turbo-shaft


The code executes an SQL query to analyze the Planes table and stores the result in a pandas DataFrame named task3_sql. 
>The query 'SELECT COUNT(*), engine FROM Planes GROUP BY engine' counts the number of rows (COUNT(*)) for each distinct engine value in the Planes table and groups the results by the engine column. This means it computes the frequency of each unique engine type in the table. <br> 
The result of this query is then read into Python as a DataFrame using pd.read_sql_query, which executes the SQL query using the specified sql_connection. <br> 
The resulting DataFrame, task3_sql, contains two columns: the first column represents the count of rows for each engine type, and the second column represents the corresponding engine type. This makes the data easy to work with for further analysis in Python.

In [15]:
task3_pd= (planes.groupby('engine')
           .size().reset_index(name='COUNT(*)')
           [['COUNT(*)','engine']])
task3_pd

,COUNT(*),engine
0,2,4 Cycle
1,28,Reciprocating
2,2750,Turbo-fan
3,535,Turbo-jet
4,2,Turbo-prop
5,5,Turbo-shaft


The code calculates the count of rows for each unique value in the engine column from the planes DataFrame and stores the result in a new DataFrame, task3_pd.<br>
>First, planes.groupby('engine') groups the data by the engine column. Then, .size() computes the size of each group, effectively counting how many rows correspond to each engine type. The result is a Series with the counts, indexed by engine. <br>
Next, .reset_index(name='COUNT(*)') converts this Series into a DataFrame, renaming the count column to COUNT(*) and resetting the index so the engine values become a regular column instead of an index. Finally, task3_pd[['COUNT(*)', 'engine']] reorders the columns to display the count (COUNT(*)) first and the engine values second, creating a neatly formatted DataFrame. <br> 
The resulting task3_pd DataFrame contains two columns: one showing the count of rows for each engine type and another listing the corresponding engine types. This structure mirrors the result of the SQL query in the earlier example, but it is done entirely using pandas.

In [16]:
print(pd.testing.assert_frame_equal(task3_sql,task3_pd))

None


### Subtask-4

In [17]:
task4_sql =pd.read_sql_query('''  SELECT COUNT(*), engine, type FROM Planes GROUP BY engine, type  ''', sql_connection)
task4_sql

,COUNT(*),engine,type
0,2,4 Cycle,Fixed wing single engine
1,5,Reciprocating,Fixed wing multi engine
2,23,Reciprocating,Fixed wing single engine
3,2750,Turbo-fan,Fixed wing multi engine
4,535,Turbo-jet,Fixed wing multi engine
5,2,Turbo-prop,Fixed wing multi engine
6,5,Turbo-shaft,Rotorcraft


The code executes an SQL query to compute the count of rows for each unique combination of engine and type in the Planes table and stores the result in a pandas DataFrame named task4_sql. <br>
>The query 'SELECT COUNT(*), engine, type FROM Planes GROUP BY engine, type' groups the rows in the Planes table by the engine and type columns, then calculates the count (COUNT(*)) of rows in each group. This means it calculates how many times each unique pair of engine and type occurs in the table. <br>
The result of this query is fetched into Python using pd.read_sql_query, which runs the SQL query and converts the output into a pandas DataFrame. <br>
The resulting task4_sql DataFrame contains three columns: the first column (COUNT(*)) shows the count of rows for each engine-type combination, while the second and third columns represent the corresponding engine and type values. This makes the data easy to analyze further in Python.

In [18]:
task4_pd= (planes.groupby(['type','engine'])
           .size().reset_index(name='COUNT(*)')
           [['COUNT(*)','engine','type']]
           .sort_values(by='engine')
           .reset_index(drop=True))
task4_pd

,COUNT(*),engine,type
0,2,4 Cycle,Fixed wing single engine
1,5,Reciprocating,Fixed wing multi engine
2,23,Reciprocating,Fixed wing single engine
3,2750,Turbo-fan,Fixed wing multi engine
4,535,Turbo-jet,Fixed wing multi engine
5,2,Turbo-prop,Fixed wing multi engine
6,5,Turbo-shaft,Rotorcraft


The code computes the count of rows for each unique combination of type and engine in the planes DataFrame and stores the result in task4_pd.<br> 
>First, planes.groupby(['type', 'engine']) groups the data by both the type and engine columns, creating groups for every unique combination of these two columns. Then, .size() calculates the size of each group, effectively counting how many rows exist for each type-engine pair. This grouped data is converted into a DataFrame using .reset_index(name='COUNT(*)'), where the counts are added as a new column named COUNT(*). <br>
Next, the code reorders the columns to display COUNT(*) first, followed by engine and type, using task4_pd[['COUNT(*)', 'engine', 'type']]. It then sorts the rows of the DataFrame by the engine column in ascending order with .sort_values(by='engine'). <br>
Finally, .reset_index(drop=True) resets the row indices to sequential numbers starting from 0, ensuring a clean DataFrame structure.<br> 
The resulting task4_pd contains three columns: the count of rows (COUNT(*)), the engine type, and the type, sorted for easier interpretation.

In [19]:
print(pd.testing.assert_frame_equal(task4_sql,task4_pd))

None


### Subtask-5

In [20]:
task5_sql =pd.read_sql_query('''  SELECT MIN(year), AVG(year), MAX(year), engine, manufacturer
FROM Planes
GROUP BY engine, manufacturer  ''', sql_connection)
task5_sql.head()

,MIN(year),AVG(year),MAX(year),engine,manufacturer
0,1975.0,1975.0,1975.0,4 Cycle,CESSNA
1,NaN,NaN,NaN,4 Cycle,JOHN G HESS
2,NaN,NaN,NaN,Reciprocating,AMERICAN AIRCRAFT INC
3,2007.0,2007.0,2007.0,Reciprocating,AVIAT AIRCRAFT INC
4,NaN,NaN,NaN,Reciprocating,BARKER JACK L


Using the pd.read_sql_query function, it executes an SQL query on a connected database via sql_connection. <br>
>The query calculates the minimum (MIN(year)), average (AVG(year)), and maximum (MAX(year)) values for the year column, grouped by the engine and manufacturer columns from the Planes table. The resulting data is loaded into a Pandas DataFrame named task5_sql. <br>
Finally, the head() method is used to display the first few rows of the DataFrame, allowing a quick inspection of the summarized data. This snippet effectively combines SQL and Pandas to perform database operations and data visualization seamlessly.

In [21]:

task5_pd= planes.groupby(['manufacturer','engine'], as_index=False).agg(
    MIN_Year=('year','min'),
    Max_Year=('year','max'),
    AVG_Year=('year','mean'))

task5_pd=task5_pd.sort_values(by=['engine','manufacturer']).reset_index(drop=True)

task5_pd=task5_pd[['MIN_Year', 'AVG_Year', 'Max_Year', 'engine','manufacturer']]

task5_pd=task5_pd.rename(columns={'MIN_Year': 'MIN(year)', 
                                  'AVG_Year': 'AVG(year)', 
                                  'Max_Year': 'MAX(year)'})
task5_pd.head()
print("task complete")

,MIN(year),AVG(year),MAX(year),engine,manufacturer
0,1975.0,1975.0,1975.0,4 Cycle,CESSNA
1,NaN,NaN,NaN,4 Cycle,JOHN G HESS
2,NaN,NaN,NaN,Reciprocating,AMERICAN AIRCRAFT INC
3,2007.0,2007.0,2007.0,Reciprocating,AVIAT AIRCRAFT INC
4,NaN,NaN,NaN,Reciprocating,BARKER JACK L


The above code begins by grouping the data based on the manufacturer and engine columns and applies aggregations to the year column to calculate the minimum (MIN_Year), maximum (Max_Year), and average (AVG_Year) year for each group. <br>
>The grouped data is then sorted by engine and manufacturer in ascending order, and the index is reset to maintain a clean, sequential index. Following this, the snippet selects specific columns of interest (MIN_Year, AVG_Year, Max_Year, engine, and manufacturer) and arranges them in a desired order.<br> 
Lastly, the columns are renamed to more descriptive labels: MIN(year), AVG(year), and MAX(year). The final DataFrame provides a concise summary of the year statistics for each combination of manufacturer and engine, making the data easier to interpret and analyze.

In [22]:
print(pd.testing.assert_frame_equal(task5_sql,task5_pd))

None


### Subtask-6

In [23]:
task6_sql =pd.read_sql_query('''  SELECT * FROM planes WHERE speed IS NOT NULL  ''', sql_connection)
task6_sql.head(6)

,tailnum,year,type,manufacturer,model,engines,seats,speed,engine
0,N201AA,1959.0,Fixed wing single engine,CESSNA,150,1,2,90.0,Reciprocating
1,N202AA,1980.0,Fixed wing multi engine,CESSNA,421C,2,8,90.0,Reciprocating
2,N350AA,1980.0,Fixed wing multi engine,PIPER,PA-31-350,2,8,162.0,Reciprocating
3,N364AA,1973.0,Fixed wing multi engine,CESSNA,310Q,2,6,167.0,Reciprocating
4,N378AA,1963.0,Fixed wing single engine,CESSNA,172E,1,4,105.0,Reciprocating
5,N381AA,1956.0,Fixed wing multi engine,DOUGLAS,DC-7BF,4,102,232.0,Reciprocating


The code retrieves all rows from the planes table where the speed column is not null and stores the result in a pandas DataFrame named task6_sql.<br> 
>The SQL query 'SELECT * FROM planes WHERE speed IS NOT NULL' selects all columns (*) from the planes table but filters the rows to include only those where the speed column has a value (i.e., it is not null). This ensures that rows with missing or null values in the speed column are excluded. <br>
The query is executed using the pd.read_sql_query function, which runs the SQL query on the database connected via sql_connection and imports the result into a pandas DataFrame. <br>
The resulting DataFrame, task6_sql, contains all columns from the planes table but only includes rows where the speed column has valid, non-null values. This makes it suitable for analyzing records with known speed data.

In [24]:
task6_pd=(planes.dropna()
          .reset_index(drop=True))
task6_pd.head(6)

,tailnum,year,type,manufacturer,model,engines,seats,speed,engine
0,N201AA,1959.0,Fixed wing single engine,CESSNA,150,1,2,90.0,Reciprocating
1,N202AA,1980.0,Fixed wing multi engine,CESSNA,421C,2,8,90.0,Reciprocating
2,N350AA,1980.0,Fixed wing multi engine,PIPER,PA-31-350,2,8,162.0,Reciprocating
3,N364AA,1973.0,Fixed wing multi engine,CESSNA,310Q,2,6,167.0,Reciprocating
4,N378AA,1963.0,Fixed wing single engine,CESSNA,172E,1,4,105.0,Reciprocating
5,N381AA,1956.0,Fixed wing multi engine,DOUGLAS,DC-7BF,4,102,232.0,Reciprocating


The code filters the planes DataFrame to remove any rows with missing (null) values in any column, and stores the result in a new DataFrame named task6_pd.<br> 
>The planes.dropna() function eliminates all rows where at least one column contains a NaN value. This ensures that task6_pd contains only rows with complete data across all columns. The .reset_index(drop=True) method resets the index of the resulting DataFrame to sequential numbers starting from 0, discarding the original index. <br>
Finally, task6_pd.head(6) displays the first 6 rows of the cleaned DataFrame, allowing you to inspect a sample of the rows with no missing values. <br>
This approach is helpful when working with clean, complete datasets for analysis.

In [25]:
print(pd.testing.assert_frame_equal(task6_sql,task6_pd))

None


### Subtask-7

In [26]:
task7_sql =pd.read_sql_query('''  SELECT tailnum FROM planes
WHERE seats BETWEEN 150 AND 210 AND year >= 2011  ''', sql_connection)
task7_sql

,tailnum
0,N150UW
1,N151UW
2,N152UW
3,N153UW
4,N154UW
...,...
87,N851VA
88,N852VA
89,N853VA
90,N854VA


The code retrieves the tailnum values from the planes table for planes that meet specific conditions and stores the result in a pandas DataFrame named task7_sql.<br> 
>The SQL query 'SELECT tailnum FROM planes WHERE seats BETWEEN 150 AND 210 AND year >= 2011' filters the data to include only those rows where the seats column has a value between 150 and 210 (inclusive) and the year column has a value greater than or equal to 2011. <br> 
The tailnum column, which typically represents the unique identifier for an aircraft, is then selected from the rows that meet these criteria. The pd.read_sql_query function executes this query on the database connected through sql_connection and imports the filtered results into a pandas DataFrame. <br>
The resulting DataFrame, task7_sql, contains a single column, tailnum, listing the unique identifiers for planes that satisfy the specified conditions. This is useful for identifying aircraft with a specific seating capacity and manufacturing year.

In [27]:
task7_pd=(planes[
            (planes['seats']>150) & 
           (planes['seats']<210) & 
           (planes['year']>=2011)][['tailnum']]
           .reset_index(drop=True))

The code filters the planes DataFrame to select rows where the seats column is between 150 and 210 (exclusive), and the year column is greater than or equal to 2011, and stores the result in a new DataFrame named task7_pd. <bt>
>The condition (planes['seats']>150) & (planes['seats']<210) & (planes['year']>=2011) uses logical operators (&) to combine these three conditions. It checks that the seats value is greater than 150 but less than 210, and that the year is 2011 or later. After filtering the rows, [['tailnum']] selects only the tailnum column from the filtered data, which typically contains unique identifiers for the aircraft. The .reset_index(drop=True) resets the index of the resulting DataFrame to sequential numbers starting from 0, while drop=True ensures the old index is discarded. <br>
The final DataFrame, task7_pd, contains the tailnum values of planes that meet the specified conditions, making it easy to inspect or work with the filtered data.

In [28]:
print(pd.testing.assert_frame_equal(task7_sql,task7_pd))

None


### Subtask-8

In [29]:
task8_sql =pd.read_sql_query('''  SELECT tailnum, manufacturer, seats FROM planes
WHERE manufacturer IN ("BOEING", "AIRBUS", "EMBRAER") AND seats>390  ''', sql_connection)
task8_sql

,tailnum,manufacturer,seats
0,N206UA,BOEING,400
1,N228UA,BOEING,400
2,N272AT,BOEING,400
3,N57016,BOEING,400
4,N670US,BOEING,450
5,N77012,BOEING,400
6,N777UA,BOEING,400
7,N78003,BOEING,400
8,N78013,BOEING,400
9,N787UA,BOEING,400


The code retrieves the tailnum, manufacturer, and seats columns from the planes table, filtering for specific manufacturers and seating capacities, and stores the result in a pandas DataFrame named task8_sql. <br> 
>The SQL query 'SELECT tailnum, manufacturer, seats FROM planes WHERE manufacturer IN ("BOEING", "AIRBUS", "EMBRAER") AND seats>390' filters the rows based on two conditions: it selects only rows where the manufacturer is one of "BOEING", "AIRBUS", or "EMBRAER", and where the seats column is greater than 390. The IN operator is used to specify the manufacturers of interest, and the condition seats>390 filters for planes with more than 390 seats. <br>
The pd.read_sql_query function executes the query on the database connected through sql_connection and retrieves the filtered result into a pandas DataFrame.<br> 
The resulting task8_sql DataFrame contains the tailnum, manufacturer, and seats for the planes that meet these criteria, making it useful for identifying large aircraft from these specific manufacturers.

In [30]:
task8_pd=(planes[
    (
    (planes['manufacturer']=="BOEING") |
    (planes['manufacturer']=="AIRBUS") |
    (planes['manufacturer']=="EMBRAER")
    ) &
    (planes['seats']>390)][['tailnum','manufacturer','seats']].reset_index(drop=True))

The code filters the planes DataFrame to select rows where the manufacturer is either "BOEING", "AIRBUS", or "EMBRAER" and the seats column is greater than 390, and stores the result in a new DataFrame named task8_pd. <br>
>The condition ((planes['manufacturer']=="BOEING") | (planes['manufacturer']=="AIRBUS") | (planes['manufacturer']=="EMBRAER")) uses the logical OR (|) operator to check if the manufacturer is one of the three specified values. 
The second condition (planes['seats']>390) filters for planes that have more than 390 seats. Both conditions are combined using the logical AND (&) operator. <br>
After filtering the rows, [['tailnum','manufacturer','seats']] selects only the relevant columns: tailnum, manufacturer, and seats. The .reset_index(drop=True) method resets the index of the resulting DataFrame to sequential numbers starting from 0, discarding the old index.<br>
The final DataFrame, task8_pd, contains the tailnum, manufacturer, and seats for the planes that meet these criteria, making it suitable for further analysis of large aircraft from the specified manufacturers.

In [31]:
print(pd.testing.assert_frame_equal(task8_sql,task8_pd))

None


### Subtask-9

In [32]:
task9_sql =pd.read_sql_query('''  SELECT DISTINCT year, seats FROM planes
WHERE year >= 2012 ORDER BY year ASC, seats DESC  ''', sql_connection)
task9_sql

,year,seats
0,2012.0,379
1,2012.0,377
2,2012.0,260
3,2012.0,222
4,2012.0,200
5,2012.0,191
6,2012.0,182
7,2012.0,149
8,2012.0,140
9,2012.0,20


The code retrieves distinct values for year and seats from the planes table, filtering for rows where the year is greater than or equal to 2012, and stores the result in a pandas DataFrame named task9_sql. <br> 
>The SQL query 'SELECT DISTINCT year, seats FROM planes WHERE year >= 2012 ORDER BY year ASC, seats DESC' performs several operations. The DISTINCT keyword ensures that only unique combinations of year and seats are selected, eliminating any duplicates. The WHERE year >= 2012 condition filters for planes manufactured from 2012 onward. The results are then ordered by year in ascending (ASC) order and by seats in descending (DESC) order.<br> 
The pd.read_sql_query function executes this SQL query on the database connected through sql_connection and imports the result into a pandas DataFrame.<br> 
The resulting task9_sql DataFrame contains the distinct year and seats combinations, sorted by year and seats, providing insight into the available plane models from 2012 and beyond, ordered by the number of seats.

In [33]:
task9_pd=(planes[((planes['year']>=2012))]
          [['year','seats']]
          .drop_duplicates()
          .sort_values(by=['year','seats'],ascending=[True,False])
          .reset_index(drop=True))
task9_pd

,year,seats
0,2012.0,379
1,2012.0,377
2,2012.0,260
3,2012.0,222
4,2012.0,200
5,2012.0,191
6,2012.0,182
7,2012.0,149
8,2012.0,140
9,2012.0,20


The code filters the planes DataFrame to include only rows where the year is greater than or equal to 2012, and then extracts distinct combinations of year and seats, storing the result in a new DataFrame named task9_pd. <br>
>The condition (planes['year']>=2012) filters the data for planes manufactured from 2012 onward. The [['year', 'seats']] part selects only the year and seats columns for further processing. The .drop_duplicates() method removes any duplicate combinations of year and seats, ensuring only unique rows remain. The .sort_values(by=['year', 'seats'], ascending=[True, False]) sorts the DataFrame first by year in ascending order (True), and then by seats in descending order (False). <br>
Finally, .reset_index(drop=True) resets the index to sequential numbers starting from 0, discarding the original index. <br>
The resulting task9_pd contains unique year and seats pairs, sorted in the specified order, making it suitable for analyzing the available planes from 2012 and beyond with their seat configurations.

In [34]:
print(pd.testing.assert_frame_equal(task9_sql,task9_pd))

None


### Subtask-10

In [35]:
task10_sql =pd.read_sql_query('''  SELECT DISTINCT year, seats FROM planes
WHERE year >= 2012 ORDER BY seats DESC, year ASC  ''', sql_connection)
task10_sql

,year,seats
0,2012.0,379
1,2013.0,379
2,2012.0,377
3,2013.0,377
4,2012.0,260
5,2012.0,222
6,2013.0,222
7,2012.0,200
8,2013.0,200
9,2013.0,199


The code executes an SQL query to retrieve distinct combinations of year and seats from the planes table, filtering for planes manufactured in 2012 or later, and stores the result in a pandas DataFrame named task10_sql. <br>
>The SQL query 'SELECT DISTINCT year, seats FROM planes WHERE year >= 2012 ORDER BY seats DESC, year ASC' performs several tasks. The DISTINCT keyword ensures that only unique pairs of year and seats are returned. The condition WHERE year >= 2012 filters the data to include only planes from 2012 onward. <br>
The results are then ordered first by seats in descending (DESC) order, ensuring that planes with the most seats appear first, and then by year in ascending (ASC) order to maintain a chronological order for planes with the same seating capacity. The pd.read_sql_query function executes this query on the database connected through sql_connection and returns the filtered and sorted results as a pandas DataFrame. <br>
The resulting task10_sql DataFrame contains the distinct year and seats combinations, sorted by the number of seats and year, providing a view of the seating configurations of planes from 2012 onward.

In [36]:
task10_pd=(planes[((planes['year']>=2012))]
           [['year','seats']] #choosing only these columns
           .drop_duplicates()
           .sort_values(by=['seats','year'],ascending=[False,True])
           .reset_index(drop=True))
task10_pd

,year,seats
0,2012.0,379
1,2013.0,379
2,2012.0,377
3,2013.0,377
4,2012.0,260
5,2012.0,222
6,2013.0,222
7,2012.0,200
8,2013.0,200
9,2013.0,199


The code filters the planes DataFrame to include only rows where the year is greater than or equal to 2012, then extracts unique combinations of year and seats, and stores the result in a new DataFrame named task10_pd. <br>
>The condition (planes['year']>=2012) filters the data to only include planes manufactured from 2012 onwards. The [['year', 'seats']] selects the year and seats columns for further processing. The .drop_duplicates() method ensures that only unique combinations of year and seats are retained, eliminating any duplicates. <br>
The .sort_values(by=['seats', 'year'], ascending=[False, True]) sorts the data first by seats in descending order (False), so planes with more seats appear first, and then by year in ascending order (True), to ensure that for planes with the same number of seats, they are listed in chronological order.<br> 
Finally, .reset_index(drop=True) resets the index to sequential numbers starting from 0, discarding the original index. The resulting task10_pd DataFrame contains the distinct year and seats pairs, sorted first by seating capacity and then by year, giving a clear view of the seating configurations of planes manufactured from 2012 onward.

In [37]:
print(pd.testing.assert_frame_equal(task10_sql,task10_pd))

None


### Subtask-11

In [38]:
task11_sql =pd.read_sql_query('''  SELECT manufacturer, COUNT(*) FROM planes
WHERE seats > 200 GROUP BY manufacturer  ''', sql_connection)
task11_sql

,manufacturer,COUNT(*)
0,AIRBUS,66
1,AIRBUS INDUSTRIE,4
2,BOEING,225


The code executes an SQL query to count the number of planes for each manufacturer where the number of seats is greater than 200, and stores the result in a pandas DataFrame named task11_sql. <br>
>The SQL query 'SELECT manufacturer, COUNT(*) FROM planes WHERE seats > 200 GROUP BY manufacturer' performs the following operations: the WHERE seats > 200 condition filters the rows to include only planes with more than 200 seats. <br>
The GROUP BY manufacturer clause groups the rows by the manufacturer column, so that the planes are categorized by their manufacturer. The COUNT(*) function counts the number of rows (planes) in each group, effectively giving the number of planes each manufacturer has with more than 200 seats. <br>
The pd.read_sql_query function executes this query on the database connected through sql_connection and retrieves the result as a pandas DataFrame. <br>
The resulting task11_sql DataFrame contains two columns: manufacturer and the corresponding count of planes with more than 200 seats for each manufacturer. This is useful for analyzing the distribution of large planes

In [39]:
task11_pd= (planes[planes['seats']>200]
            .groupby('manufacturer')
            .size().reset_index(name='COUNT(*)'))
task11_pd

,manufacturer,COUNT(*)
0,AIRBUS,66
1,AIRBUS INDUSTRIE,4
2,BOEING,225


The code filters the planes DataFrame to include only planes with more than 200 seats, then groups the data by manufacturer and calculates the count of such planes for each manufacturer, storing the result in a new DataFrame named task11_pd. <br>
>The condition planes['seats']>200 filters the rows, selecting only planes with more than 200 seats. The .groupby('manufacturer') method groups the filtered data by the manufacturer column, so that the planes are grouped based on their manufacturer. The .size() function counts the number of planes in each group (i.e., the number of planes per manufacturer with more than 200 seats). <br>
The .reset_index(name='COUNT(*)') method resets the index of the resulting grouped data and renames the count column to COUNT(*). <br>
The resulting task11_pd DataFrame contains two columns: manufacturer and COUNT(*), representing the manufacturer and the count of planes with more than 200 seats for each manufacturer. This approach provides a summary of the distribution of large planes across different manufacturers.

In [40]:
print(pd.testing.assert_frame_equal(task11_sql,task11_pd))

None


### Subtask-12

In [41]:
task12_sql =pd.read_sql_query('''  SELECT manufacturer, COUNT(*) FROM planes
GROUP BY manufacturer HAVING COUNT(*) > 10  ''', sql_connection)
task12_sql

,manufacturer,COUNT(*)
0,AIRBUS,336
1,AIRBUS INDUSTRIE,400
2,BOEING,1630
3,BOMBARDIER INC,368
4,EMBRAER,299
5,MCDONNELL DOUGLAS,120
6,MCDONNELL DOUGLAS AIRCRAFT CO,103
7,MCDONNELL DOUGLAS CORPORATION,14


The code executes an SQL query to retrieve the count of planes for each manufacturer, but only for those manufacturers that have more than 10 planes in total, and stores the result in a pandas DataFrame named task12_sql.<br> 
>The SQL query 'SELECT manufacturer, COUNT(*) FROM planes GROUP BY manufacturer HAVING COUNT(*) > 10' performs several operations. The GROUP BY manufacturer clause groups the planes by their manufacturer, so each group represents all planes made by a particular manufacturer. <br>
The COUNT(*) function counts the number of planes in each group. The HAVING COUNT(*) > 10 condition filters the results to only include those manufacturers that have more than 10 planes. This is different from a WHERE clause because it filters after the grouping operation. The pd.read_sql_query function executes this query on the database connected through sql_connection and retrieves the result as a pandas DataFrame. <br>
The resulting task12_sql DataFrame contains two columns: manufacturer and the corresponding count of planes for those manufacturers with more than 10 planes. This query is useful for identifying the major manufacturers with a significant number of planes in the dataset.

In [42]:
task12_pd= planes.groupby('manufacturer').size().reset_index(name='COUNT(*)')
task12_pd=task12_pd[task12_pd['COUNT(*)']>10].reset_index(drop=True)
task12_pd

,manufacturer,COUNT(*)
0,AIRBUS,336
1,AIRBUS INDUSTRIE,400
2,BOEING,1630
3,BOMBARDIER INC,368
4,EMBRAER,299
5,MCDONNELL DOUGLAS,120
6,MCDONNELL DOUGLAS AIRCRAFT CO,103
7,MCDONNELL DOUGLAS CORPORATION,14


The code groups the planes DataFrame by manufacturer, counts the number of planes for each manufacturer, and filters to retain only those manufacturers with more than 10 planes, storing the result in a new DataFrame named task12_pd. <br>
>The .groupby('manufacturer') method groups the data by the manufacturer column, so each group represents all planes made by a particular manufacturer. The .size() function counts the number of planes in each group, and .reset_index(name='COUNT(*)') resets the index and renames the count column to COUNT(*). The resulting DataFrame task12_pd contains two columns: manufacturer and COUNT(*). <br>
The condition task12_pd['COUNT(*)']>10 filters the DataFrame to include only those rows where the count of planes is greater than 10. Finally, .reset_index(drop=True) resets the index of the filtered DataFrame to sequential numbers starting from 0, discarding the old index. <br>
The resulting task12_pd contains manufacturers with more than 10 planes in the dataset, providing insights into the most significant manufacturers in terms of the number of planes.

In [43]:
print(pd.testing.assert_frame_equal(task12_sql,task12_pd))

None


### Subtask-13

In [44]:
task13_sql =pd.read_sql_query('''  SELECT manufacturer, COUNT(*) FROM planes
WHERE seats > 200 GROUP BY manufacturer HAVING COUNT(*) > 10  ''', sql_connection)
task13_sql

,manufacturer,COUNT(*)
0,AIRBUS,66
1,BOEING,225


The code executes an SQL query to retrieve the count of planes with more than 200 seats for each manufacturer, but only for manufacturers that have more than 10 such planes, and stores the result in a pandas DataFrame named task13_sql. <br>
>The SQL query 'SELECT manufacturer, COUNT(*) FROM planes WHERE seats > 200 GROUP BY manufacturer HAVING COUNT(*) > 10' performs several operations. First, the WHERE seats > 200 condition filters the data to include only planes with more than 200 seats. Then, the GROUP BY manufacturer clause groups the data by the manufacturer column, so that each group represents all planes made by a particular manufacturer. <br>
The COUNT(*) function counts the number of planes in each group that meet the seat condition. Finally, the HAVING COUNT(*) > 10 condition filters the grouped results to include only those manufacturers with more than 10 planes that have more than 200 seats.<br> 
The pd.read_sql_query function executes this query on the database connected through sql_connection and returns the result as a pandas DataFrame. <br>
The resulting task13_sql DataFrame contains the manufacturer and the corresponding count of planes with more than 200 seats for those manufacturers that have more than 10 such planes. This helps identify manufacturers with a significant number of large aircraft.

In [45]:
task13_pd=planes[planes['seats']>200].groupby('manufacturer').size().reset_index(name='COUNT(*)')
task13_pd=task13_pd[task13_pd['COUNT(*)']>10].reset_index(drop=True)
task13_pd

,manufacturer,COUNT(*)
0,AIRBUS,66
1,BOEING,225


The code filters the planes DataFrame to include only planes with more than 200 seats, then groups the data by manufacturer, counts the number of such planes for each manufacturer, and filters out manufacturers with 10 or fewer planes, storing the result in a new DataFrame named task13_pd. <br>
>The condition planes['seats']>200 filters the rows to select only planes with more than 200 seats. The .groupby('manufacturer') method groups the filtered data by the manufacturer column, and the .size() function counts the number of planes in each group. The .reset_index(name='COUNT(*)') method resets the index and names the count column COUNT(*). <br>
The task13_pd[task13_pd['COUNT(*)']>10] part filters the grouped data to include only those manufacturers with more than 10 planes. Finally, .reset_index(drop=True) resets the index of the filtered DataFrame to sequential numbers starting from 0, discarding the old index. <br>
The resulting task13_pd DataFrame contains the manufacturer and the corresponding count of planes with more than 200 seats, but only for manufacturers with more than 10 such planes. This helps identify major manufacturers with a significant number of large aircraft.

In [46]:
print(pd.testing.assert_frame_equal(task13_sql,task13_pd))

None


### Subtask-14

In [47]:
task14_sql =pd.read_sql_query('''  SELECT manufacturer, COUNT(*) AS howmany
FROM planes
GROUP BY manufacturer
ORDER BY howmany DESC LIMIT 10  ''', sql_connection)
task14_sql

,manufacturer,howmany
0,BOEING,1630
1,AIRBUS INDUSTRIE,400
2,BOMBARDIER INC,368
3,AIRBUS,336
4,EMBRAER,299
5,MCDONNELL DOUGLAS,120
6,MCDONNELL DOUGLAS AIRCRAFT CO,103
7,MCDONNELL DOUGLAS CORPORATION,14
8,CESSNA,9
9,CANADAIR,9


The code executes an SQL query to retrieve the top 10 manufacturers with the highest number of planes, and stores the result in a pandas DataFrame named task14_sql. <br>
>The SQL query 'SELECT manufacturer, COUNT(*) AS howmany FROM planes GROUP BY manufacturer ORDER BY howmany DESC LIMIT 10' performs the following operations: The GROUP BY manufacturer clause groups the planes by their manufacturer, so each group represents all planes made by a particular manufacturer. The COUNT(*) AS howmany counts the number of planes in each group and renames the count column to howmany. <br>
The ORDER BY howmany DESC sorts the results in descending order based on the count, so that the manufacturers with the most planes appear first. <br>
Finally, the LIMIT 10 clause restricts the result to only the top 10 manufacturers with the highest plane counts. The pd.read_sql_query function executes this query on the database connected through sql_connection and retrieves the result as a pandas DataFrame. <br>
The resulting task14_sql DataFrame contains two columns: manufacturer and howmany, which represent the top 10 manufacturers and the number of planes they have, ordered from the most to the least planes.

In [48]:
task14_pd=(planes.groupby('manufacturer')
           .size().reset_index(name='howmany')
           .sort_values(by='howmany',ascending=False)
           .head(10)
           .reset_index(drop=True))
task14_pd

,manufacturer,howmany
0,BOEING,1630
1,AIRBUS INDUSTRIE,400
2,BOMBARDIER INC,368
3,AIRBUS,336
4,EMBRAER,299
5,MCDONNELL DOUGLAS,120
6,MCDONNELL DOUGLAS AIRCRAFT CO,103
7,MCDONNELL DOUGLAS CORPORATION,14
8,CESSNA,9
9,CANADAIR,9


The code groups the planes DataFrame by manufacturer, counts the number of planes for each manufacturer, sorts the manufacturers by the number of planes in descending order, and retrieves the top 10 manufacturers, storing the result in a new DataFrame named task14_pd. <br>
>The .groupby('manufacturer') method groups the data by the manufacturer column, so each group represents all planes made by a particular manufacturer. The .size() function counts the number of planes in each group, and .reset_index(name='howmany') resets the index and names the count column howmany. <br>
The .sort_values(by='howmany', ascending=False) sorts the resulting DataFrame by the howmany column in descending order, so that manufacturers with the most planes appear first. The .head(10) method selects the top 10 rows, giving the 10 manufacturers with the highest number of planes. Finally, .reset_index(drop=True) resets the index of the filtered DataFrame to sequential numbers starting from 0, discarding the original index. <br>
The resulting task14_pd DataFrame contains the manufacturer and howmany columns, representing the top 10 manufacturers and the corresponding count of planes, ordered from the most to the least planes.

In [49]:
print(pd.testing.assert_frame_equal(task14_sql,task14_pd))

None


### Subtask-15

In [50]:
task15_sql =pd.read_sql_query('''  SELECT flights.*, planes.year AS plane_year, planes.speed AS plane_speed,
planes.seats AS plane_seats
FROM flights LEFT JOIN planes ON flights.tailnum=planes.tailnum  ''', sql_connection)
task15_sql

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,...,origin,dest,air_time,distance,hour,minute,time_hour,plane_year,plane_speed,plane_seats
0,2013,1,1,517.0,515,2.0,830.0,819,11.0,UA,...,EWR,IAH,227.0,1400,5,15,2013-01-01 05:00:00,1999.0,NaN,149.0
1,2013,1,1,533.0,529,4.0,850.0,830,20.0,UA,...,LGA,IAH,227.0,1416,5,29,2013-01-01 05:00:00,1998.0,NaN,149.0
2,2013,1,1,542.0,540,2.0,923.0,850,33.0,AA,...,JFK,MIA,160.0,1089,5,40,2013-01-01 05:00:00,1990.0,NaN,178.0
3,2013,1,1,544.0,545,-1.0,1004.0,1022,-18.0,B6,...,JFK,BQN,183.0,1576,5,45,2013-01-01 05:00:00,2012.0,NaN,200.0
4,2013,1,1,554.0,600,-6.0,812.0,837,-25.0,DL,...,LGA,ATL,116.0,762,6,0,2013-01-01 06:00:00,1991.0,NaN,178.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
336771,2013,9,30,NaN,1455,NaN,NaN,1634,NaN,9E,...,JFK,DCA,NaN,213,14,55,2013-09-30 14:00:00,NaN,NaN,NaN
336772,2013,9,30,NaN,2200,NaN,NaN,2312,NaN,9E,...,LGA,SYR,NaN,198,22,0,2013-09-30 22:00:00,NaN,NaN,NaN
336773,2013,9,30,NaN,1210,NaN,NaN,1330,NaN,MQ,...,LGA,BNA,NaN,764,12,10,2013-09-30 12:00:00,NaN,NaN,NaN
336774,2013,9,30,NaN,1159,NaN,NaN,1344,NaN,MQ,...,LGA,CLE,NaN,419,11,59,2013-09-30 11:00:00,NaN,NaN,NaN


The code executes an SQL query to retrieve detailed flight data along with additional information from the planes table, such as the year, speed, and seats of the plane associated with each flight. The result is stored in a pandas DataFrame named task15_sql. <br>
>The SQL query 'SELECT flights.*, planes.year AS plane_year, planes.speed AS plane_speed, planes.seats AS plane_seats FROM flights LEFT JOIN planes ON flights.tailnum=planes.tailnum' performs several operations. The SELECT flights.* retrieves all columns from the flights table. <br>
The planes.year AS plane_year, planes.speed AS plane_speed, and planes.seats AS plane_seats select the year, speed, and seats columns from the planes table, and rename them for clarity. <br>
The LEFT JOIN planes ON flights.tailnum=planes.tailnum joins the flights table with the planes table based on the tailnum column, ensuring that all rows from the flights table are included, even if there is no matching tailnum in the planes table (this is the purpose of the LEFT JOIN). <br>
The pd.read_sql_query function executes this query on the database connected through sql_connection and stores the result in the task15_sql DataFrame. <br>
This DataFrame now contains all the flight information along with additional details about the corresponding plane (year, speed, and seats).

In [51]:
task15_pd=flights.merge(planes[['year','speed','seats','tailnum']],left_on="tailnum",right_on='tailnum',how='left')
task15_pd=task15_pd.rename(columns={'year_x': 'year', 
                                  'year_y': 'plane_year', 
                                  'speed': 'plane_speed',
                                  'seats': 'plane_seats'})

The code merges the flights DataFrame with selected columns from the planes DataFrame, specifically the year, speed, seats, and tailnum columns, and stores the result in a new DataFrame named task15_pd. <br>
>The .merge() function is used to combine the two DataFrames on the tailnum column, with left_on="tailnum" specifying that the join is based on the tailnum column in the flights DataFrame, and right_on='tailnum' specifying the same column in the planes DataFrame. The how='left' parameter ensures a left join, meaning that all rows from flights will be kept, even if there is no matching tailnum in planes. <br>
After the merge, the .rename() function is used to clarify column names: year_x (from the flights DataFrame) is renamed to year, year_y (from the planes DataFrame) is renamed to plane_year, speed is renamed to plane_speed, and seats is renamed to plane_seats. <br>
The resulting task15_pd DataFrame contains all the original flights columns, along with the additional columns plane_year, plane_speed, and plane_seats from the planes table, giving detailed flight data along with the relevant plane information.

In [52]:
print(pd.testing.assert_frame_equal(task15_sql,task15_pd))

None


### Subtask-16

In [53]:
task16_sql =pd.read_sql_query('''  SELECT planes.*, airlines.* FROM
(SELECT DISTINCT carrier, tailnum FROM flights) AS cartail
INNER JOIN planes ON cartail.tailnum=planes.tailnum
INNER JOIN airlines ON cartail.carrier=airlines.carrier  ''', sql_connection)
task16_sql

,tailnum,year,type,manufacturer,model,engines,seats,speed,engine,carrier,name
0,N10156,2004.0,Fixed wing multi engine,EMBRAER,EMB-145XR,2,55,NaN,Turbo-fan,EV,ExpressJet Airlines Inc.
1,N102UW,1998.0,Fixed wing multi engine,AIRBUS INDUSTRIE,A320-214,2,182,NaN,Turbo-fan,US,US Airways Inc.
2,N103US,1999.0,Fixed wing multi engine,AIRBUS INDUSTRIE,A320-214,2,182,NaN,Turbo-fan,US,US Airways Inc.
3,N104UW,1999.0,Fixed wing multi engine,AIRBUS INDUSTRIE,A320-214,2,182,NaN,Turbo-fan,US,US Airways Inc.
4,N10575,2002.0,Fixed wing multi engine,EMBRAER,EMB-145LR,2,55,NaN,Turbo-fan,EV,ExpressJet Airlines Inc.
...,...,...,...,...,...,...,...,...,...,...,...
3334,N997AT,2002.0,Fixed wing multi engine,BOEING,717-200,2,100,NaN,Turbo-fan,FL,AirTran Airways Corporation
3335,N997DL,1992.0,Fixed wing multi engine,MCDONNELL DOUGLAS AIRCRAFT CO,MD-88,2,142,NaN,Turbo-fan,DL,Delta Air Lines Inc.
3336,N998AT,2002.0,Fixed wing multi engine,BOEING,717-200,2,100,NaN,Turbo-fan,FL,AirTran Airways Corporation
3337,N998DL,1992.0,Fixed wing multi engine,MCDONNELL DOUGLAS CORPORATION,MD-88,2,142,NaN,Turbo-jet,DL,Delta Air Lines Inc.


The code executes an SQL query to retrieve detailed information from the planes and airlines tables, based on the carriers and tail numbers present in the flights table, and stores the result in a pandas DataFrame named task16_sql. <br>
>The query 'SELECT planes.*, airlines.* FROM (SELECT DISTINCT carrier, tailnum FROM flights) AS cartail INNER JOIN planes ON cartail.tailnum=planes.tailnum INNER JOIN airlines ON cartail.carrier=airlines.carrier' performs several operations. First, the subquery (SELECT DISTINCT carrier, tailnum FROM flights) retrieves unique pairs of carrier and tailnum from the flights table, which represent the distinct combinations of airlines and planes. This result is aliased as cartail. <br>
Then, the query performs an INNER JOIN with the planes table on the tailnum column, bringing in detailed information about the planes associated with the distinct tailnum values. <br>
Another INNER JOIN is performed with the airlines table on the carrier column, bringing in the details about the airlines associated with the distinct carriers. <br>
The final result includes all columns from both planes and airlines, and it is stored in task16_sql. This DataFrame provides a detailed view of the planes and airlines associated with the carriers and tail numbers present in the flights table.

In [54]:
cartail=flights[['carrier','tailnum']].drop_duplicates().reset_index(drop=True)
cartail=cartail.sort_values(by='carrier',ascending=True)
temp1=planes.merge(cartail,how='inner')
task16_pd=temp1.merge(airlines,how='inner')
task16_pd=task16_pd

>The code first creates a DataFrame cartail containing unique combinations of carrier and tailnum from the flights DataFrame, and then uses this to merge with the planes and airlines DataFrames to gather detailed information. <br>
The expression flights[['carrier','tailnum']].drop_duplicates().reset_index(drop=True) selects the carrier and tailnum columns from the flights DataFrame, removes any duplicates, and resets the index to ensure a clean DataFrame. This DataFrame is then sorted by the carrier column in ascending order using .sort_values(by='carrier',ascending=True). <br>
The temp1 DataFrame is created by performing an inner join between planes and cartail using .merge(cartail, how='inner'), which merges the plane details with the carrier and tail number pairs. <br>
After that, the task16_pd DataFrame is created by merging temp1 with the airlines DataFrame using .merge(airlines, how='inner'), adding the airline details. <br>
The final task16_pd DataFrame contains a combination of columns from planes, cartail, and airlines, providing a detailed dataset of planes, their carriers, and associated airline information.

In [55]:
print(pd.testing.assert_frame_equal(task16_sql,task16_pd))

None


### Subtask-17

In [56]:
task17_sql =pd.read_sql_query('''  SELECT flights2.*, atemp, ahumid
FROM (
SELECT * FROM flights WHERE origin='EWR'
) AS flights2 LEFT JOIN (
SELECT year, month, day, AVG(temp) AS atemp,
AVG(humid) AS ahumid
FROM weather
WHERE origin='EWR'
GROUP BY year, month, day ) AS weather2
ON flights2.year=weather2.year
AND flights2.month=weather2.month
AND flights2.day=weather2.day  ''', sql_connection)
task17_sql

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,...,tailnum,origin,dest,air_time,distance,hour,minute,time_hour,atemp,ahumid
0,2013,1,1,517.0,515,2.0,830.0,819,11.0,UA,...,N14228,EWR,IAH,227.0,1400,5,15,2013-01-01 05:00:00,38.4800,58.386087
1,2013,1,1,554.0,558,-4.0,740.0,728,12.0,UA,...,N39463,EWR,ORD,150.0,719,5,58,2013-01-01 05:00:00,38.4800,58.386087
2,2013,1,1,555.0,600,-5.0,913.0,854,19.0,B6,...,N516JB,EWR,FLL,158.0,1065,6,0,2013-01-01 06:00:00,38.4800,58.386087
3,2013,1,1,558.0,600,-2.0,923.0,937,-14.0,UA,...,N53441,EWR,SFO,361.0,2565,6,0,2013-01-01 06:00:00,38.4800,58.386087
4,2013,1,1,559.0,600,-1.0,854.0,902,-8.0,UA,...,N76515,EWR,LAS,337.0,2227,6,0,2013-01-01 06:00:00,38.4800,58.386087
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
120830,2013,9,30,2142.0,2129,13.0,2250.0,2239,11.0,EV,...,N12957,EWR,PWM,47.0,284,21,29,2013-09-30 21:00:00,62.9075,69.806250
120831,2013,9,30,2149.0,2156,-7.0,2245.0,2308,-23.0,UA,...,N813UA,EWR,BOS,37.0,200,21,56,2013-09-30 21:00:00,62.9075,69.806250
120832,2013,9,30,2150.0,2159,-9.0,2250.0,2306,-16.0,EV,...,N10575,EWR,MHT,39.0,209,21,59,2013-09-30 21:00:00,62.9075,69.806250
120833,2013,9,30,2211.0,2059,72.0,2339.0,2242,57.0,EV,...,N12145,EWR,STL,120.0,872,20,59,2013-09-30 20:00:00,62.9075,69.806250


The code executes an SQL query to retrieve flight data from the flights table along with the corresponding daily weather information (temperature and humidity) for the origin airport EWR, and stores the result in a pandas DataFrame named task17_sql. <br>
>The SQL query 'SELECT flights2.*, atemp, ahumid FROM (SELECT * FROM flights WHERE origin='EWR') AS flights2 LEFT JOIN (SELECT year, month, day, AVG(temp) AS atemp, AVG(humid) AS ahumid FROM weather WHERE origin='EWR' GROUP BY year, month, day) AS weather2 ON flights2.year=weather2.year AND flights2.month=weather2.month AND flights2.day=weather2.day' performs several operations:<br>
Subquery for flights: The subquery (SELECT * FROM flights WHERE origin='EWR') AS flights2 selects all flight records from the flights table where the origin is EWR (Newark airport).<br>
Subquery for weather data: The subquery (SELECT year, month, day, AVG(temp) AS atemp, AVG(humid) AS ahumid FROM weather WHERE origin='EWR' GROUP BY year, month, day) AS weather2 calculates the average temperature (atemp) and humidity (ahumid) for each day at the EWR origin airport by grouping the weather data by year, month, and day.
LEFT JOIN: The two subqueries are combined using a LEFT JOIN on the year, month, and day columns, ensuring that for each flight, the corresponding average temperature and humidity values for that day are included.<br>
The final result is stored in the task17_sql DataFrame, which contains all columns from the flights data (for EWR origin) along with the atemp and ahumid columns from the weather data, providing a detailed dataset of flights and the corresponding daily weather conditions.

In [57]:
flights2=flights[flights['origin'] == 'EWR']

weather2 = weather[weather['origin'] == 'EWR'].groupby(['year', 'month', 'day']).agg(
    atemp=('temp', 'mean'),
    ahumid=('humid', 'mean')
).reset_index()

task17_pd = pd.merge(flights2, weather2, on=['year', 'month', 'day'], how='left')
task17_pd

,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,...,tailnum,origin,dest,air_time,distance,hour,minute,time_hour,atemp,ahumid
0,2013,1,1,517.0,515,2.0,830.0,819,11.0,UA,...,N14228,EWR,IAH,227.0,1400,5,15,2013-01-01 05:00:00,38.4800,58.386087
1,2013,1,1,554.0,558,-4.0,740.0,728,12.0,UA,...,N39463,EWR,ORD,150.0,719,5,58,2013-01-01 05:00:00,38.4800,58.386087
2,2013,1,1,555.0,600,-5.0,913.0,854,19.0,B6,...,N516JB,EWR,FLL,158.0,1065,6,0,2013-01-01 06:00:00,38.4800,58.386087
3,2013,1,1,558.0,600,-2.0,923.0,937,-14.0,UA,...,N53441,EWR,SFO,361.0,2565,6,0,2013-01-01 06:00:00,38.4800,58.386087
4,2013,1,1,559.0,600,-1.0,854.0,902,-8.0,UA,...,N76515,EWR,LAS,337.0,2227,6,0,2013-01-01 06:00:00,38.4800,58.386087
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
120830,2013,9,30,2142.0,2129,13.0,2250.0,2239,11.0,EV,...,N12957,EWR,PWM,47.0,284,21,29,2013-09-30 21:00:00,62.9075,69.806250
120831,2013,9,30,2149.0,2156,-7.0,2245.0,2308,-23.0,UA,...,N813UA,EWR,BOS,37.0,200,21,56,2013-09-30 21:00:00,62.9075,69.806250
120832,2013,9,30,2150.0,2159,-9.0,2250.0,2306,-16.0,EV,...,N10575,EWR,MHT,39.0,209,21,59,2013-09-30 21:00:00,62.9075,69.806250
120833,2013,9,30,2211.0,2059,72.0,2339.0,2242,57.0,EV,...,N12145,EWR,STL,120.0,872,20,59,2013-09-30 20:00:00,62.9075,69.806250


The code replicates the SQL query in Python using the flights and weather DataFrames to merge flight data with corresponding daily weather information for the origin airport EWR. <br>
>First, the flights2 DataFrame is created by filtering the flights DataFrame to include only rows where the origin is 'EWR'. <br>
Next, the weather2 DataFrame is created by filtering the weather DataFrame for records where the origin is 'EWR', then grouping the data by year, month, and day. <br>
The .agg() function is used to calculate the average temperature (atemp) and humidity (ahumid) for each day, with .reset_index() to flatten the result back into a regular DataFrame. <br>
Finally, the pd.merge() function is used to perform a left join between flights2 and weather2 based on the year, month, and day columns. The how='left' argument ensures that all records from flights2 are included, even if there is no matching weather data for a particular day. <br>
The resulting task17_pd DataFrame contains all the flight information for EWR along with the corresponding average temperature and humidity for each day, effectively combining flight and weather data.

In [58]:
print(pd.testing.assert_frame_equal(task17_sql,task17_pd))

None


In [59]:
sql_connection.close()